**问题2分析：**
当当网销量排名前 50 的 Python 类书籍，主要涵盖了哪些领域？比如，编程技巧、数据分析、机器学习和深度
学习、青少年编程等 。

In [1]:
# =============================================================
# Tsk2 销量前 50 的 Python 类图书
# =============================================================

import pandas as pd
import re
import os

# -------------------------- 1. 配置参数：领域-关键词映射（可根据需求增删） --------------------------
FIELD_KEYWORDS = {
    "Python基础入门": ["入门", "从入门到实践", "从入门到精通", "笨办法", "你好!Python", "完全自学教程"],
    "机器学习与深度学习": ["机器学习", "深度学习", "神经网络", "scikit-learn", "AI+", "自然语言处理", "强化学习"],
    "数据分析与可视化": ["数据分析", "数据科学", "数据可视化", "Pandas", "NumPy", "Matplotlib", "数亦有道"],
    "办公自动化": ["办公自动化", "Python+Office", "ChatGPT办公", "玩转Excel", "高效办公"],
    "量化交易/金融分析": ["量化交易", "金融量化", "区块链量化", "vn.py", "交易策略", "量化投资"],
    "网络爬虫": ["网络爬虫", "爬虫", "Urllib", "BeautifulSoup", "Scrapy"],
    "青少年/趣味编程": ["小学生", "青少年", "趣味编程", "看漫画学Python", "创意编程"],
    "高阶开发与配套基础": ["全栈开发", "Web编程", "现代Python编程", "矩阵", "线性代数", "程序是怎样跑起来", "智能优化算法"]
}

# -------------------------- 2. 加载CSV数据 --------------------------
# 源数据路径：相对路径 ../data_clean
csv_file = r"../data_clean/dangdang_python_books_clean.csv"
try:
    # 优先尝试utf-8编码，失败则用gbk（中文CSV常见编码）
    df = pd.read_csv(csv_file, encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(csv_file, encoding="gbk")

# 填充空值（避免匹配时报错）
df["书名"] = df["书名"].fillna("")
df["简介"] = df["简介"].fillna("")
# 合并书名+简介为匹配文本
df["match_text"] = df["书名"] + " " + df["简介"]
# 统一为小写（取消大小写影响）
df["match_text"] = df["match_text"].str.lower()

# -------------------------- 3. 定义分类匹配函数 --------------------------
def classify_book(text):
    """根据文本匹配返回所属分类，无匹配则返回「其他/跨领域融合」"""
    text = text.lower()
    for field, keywords in FIELD_KEYWORDS.items():
        for kw in keywords:
            if re.search(kw.lower(), text):
                return field
    return "其他/跨领域融合"

# 为每本书添加分类标签
df["所属分类"] = df["match_text"].apply(classify_book)

# -------------------------- 4. 归总分类信息 --------------------------
classify_summary = df.groupby("所属分类").agg(
    图书数量=("书名", "count"),
    图书列表=("书名", list),
    代表作者=("作者", lambda x: list(x.dropna().unique())[:3]),
    均价=("折后价", lambda x: round(x.dropna().mean(), 2))
).reset_index()

# -------------------------- 5. 表格形式展示结果（核心修改） --------------------------
print("="*100)
print("📊 Python图书分类归总结果（表格展示）")
print("="*100)

# 美化表格：把列表转成易读的字符串格式
summary_table = classify_summary.copy()
summary_table["图书列表"] = summary_table["图书列表"].apply(lambda x: " | ".join(x[:5]) + ("..." if len(x) > 5 else ""))
summary_table["代表作者"] = summary_table["代表作者"].apply(lambda x: "、".join(x) if x else "无")

# 输出美观表格
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 80)
print(summary_table.to_string(index=False))

# -------------------------- 6. 保存结果为CSV --------------------------
# 自动创建输出文件夹（防止路径不存在报错）
output_dir = r"../output/Tsk2"
os.makedirs(output_dir, exist_ok=True)

# 保存分类后明细数据
detail_csv = os.path.join(output_dir, "python_books_classified.csv")
df.to_csv(detail_csv, index=False, encoding="utf-8-sig")

# 保存分类汇总数据
summary_csv = os.path.join(output_dir, "python_books_classify_summary.csv")
classify_summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")

print("\n" + "="*100)
print(f"✅ 分类结果已保存：")
print(f"  1. 明细数据：{detail_csv}")
print(f"  2. 汇总数据：{summary_csv}")

FileNotFoundError: [Errno 2] No such file or directory: '../data_clean/dangdang_python_books_clean.csv'